# 🎯 Pokémon Kaggle Competition: Baseline Model (PyTorch)
Welcome to the Pokémon Kaggle Competition! 
In this notebook, we will build a **Multi-Modal Neural Network** using PyTorch. 
Our goal is to predict if a Pokémon is **Legendary** (1 = Yes, 0 = No) by combining two types of data:
1. **Numerical Stats:** (HP, Attack, Defense, etc.)
2. **Image Data:** (The official sprite of the Pokémon)

Let's build a Neural Network that handles both!

In [ ]:
import pandas as pd
import numpy as np
import os
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.preprocessing import StandardScaler

## 1. Prepare Dataset Class
In PyTorch, we need a custom `Dataset` class to load both images and numerical data simultaneously.

In [ ]:
# Numerical features we want to use
num_features = ['HP', 'Attack', 'Defense', 'Sp_Atk', 'Sp_Def', 'Speed', 'Weight', 'Height']

class PokemonDataset(Dataset):
    def __init__(self, csv_file, img_dir, scaler=None, is_train=True):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.is_train = is_train
        
        # Scale numerical features
        self.scaler = scaler
        if self.scaler is None:
            self.scaler = StandardScaler()
            self.scaled_num = self.scaler.fit_transform(self.df[num_features])
        else:
            self.scaled_num = self.scaler.transform(self.df[num_features])
            
        self.transform = transforms.Compose([
            transforms.Resize((64, 64)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # 1. Load Image
        img_name = os.path.join(self.img_dir, self.df.iloc[idx]['Image_File'])
        image = Image.open(img_name).convert('RGB')
        image = self.transform(image)
        
        # 2. Load Stats
        stats = torch.tensor(self.scaled_num[idx], dtype=torch.float32)
        
        if self.is_train:
            label = torch.tensor(self.df.iloc[idx]['Is_Legendary'], dtype=torch.float32)
            return image, stats, label
        else:
            return image, stats

# Create Datasets and DataLoaders
train_dataset = PokemonDataset('data/train.csv', 'data/images/')
test_dataset = PokemonDataset('data/test.csv', 'data/images/', scaler=train_dataset.scaler, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
print(f"Train size: {len(train_dataset)} | Test size: {len(test_dataset)}")

## 2. Build the Multi-Modal Neural Network
We will create a custom `nn.Module`.
- **Image Branch:** Flattens the image and passes it through Dense (Linear) layers.
- **Stats Branch:** Passes the numerical stats through a Linear layer.
- **Merge:** Concatenates both outputs to make a final prediction.

In [ ]:
class MultiModalNN(nn.Module):
    def __init__(self, num_stats_features):
        super(MultiModalNN, self).__init__()
        
        # Image branch (64x64x3 = 12288)
        self.img_branch = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 64 * 3, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )
        
        # Stats branch
        self.stats_branch = nn.Sequential(
            nn.Linear(num_stats_features, 32),
            nn.ReLU()
        )
        
        # Merged classification head
        self.classifier = nn.Sequential(
            nn.Linear(64 + 32, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid() # Output probability between 0 and 1
        )
        
    def forward(self, image, stats):
        img_features = self.img_branch(image)
        stats_features = self.stats_branch(stats)
        
        # Concatenate features
        merged = torch.cat((img_features, stats_features), dim=1)
        output = self.classifier(merged)
        return output

model = MultiModalNN(num_stats_features=len(num_features))
criterion = nn.BCELoss() # Binary Cross Entropy
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(model)

In [ ]:
# Train the model
epochs = 15
model.train()

for epoch in range(epochs):
    epoch_loss = 0
    correct = 0
    total = 0
    
    for images, stats, labels in train_loader:
        optimizer.zero_grad()
        
        outputs = model(images, stats).squeeze()
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        # Calculate accuracy
        preds = (outputs > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss/len(train_loader):.4f} | Acc: {correct/total:.4f}")

## 3. Generate Submission File
Now we predict on the test set and save the results into the Kaggle format.

In [ ]:
# Predict on test set
model.eval()
test_preds = []

with torch.no_grad():
    for images, stats in test_loader:
        outputs = model(images, stats).squeeze()
        
        # Handle edge case where batch size is 1 (outputs becomes scalar)
        if outputs.dim() == 0:
            outputs = outputs.unsqueeze(0)
            
        preds = (outputs > 0.5).int().numpy()
        test_preds.extend(preds)

# Prepare submission dataframe
submission = pd.read_csv('data/sample_submission.csv')
submission['Is_Legendary'] = test_preds

# Save to CSV
submission.to_csv('submission.csv', index=False)
print("submission.csv successfully created! Ready to upload to Kaggle.")